# 09 — LEAR

The **LASSO-Estimated AutoRegressive** model, the open-access benchmark of Lago, Marcjasz, De Schutter & Weron (2021): 24 hourly LASSO regressions on 246 columns (full daily price profiles of D-1, D-2, D-3, D-7; load and residual-load profiles; day of week), with an `asinh` variance-stabilising transform and penalty chosen by AIC.

- **Input:** dataset (05) and naive forecasts (06)
- **Output:** `data/processed/lear_validation_forecasts.parquet`, `results/09_lear_*`
- **Chosen model:** `lear_v1_avg` (expanding + 365-day window average) — validation MAE 9.03 €/MWh.
- **Key findings:** best RMSE **without any fuel price**: 96 lagged prices already contain the gas. Its errors are the least correlated with the other two models — the argument for an ensemble.

In [1]:
# --- Make src/ importable and reload it automatically when it changes ---------
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import forecasting as fc

TARGET = fc.TARGET

# --- Load dataset, naive forecasts and the reference models of 07 / 08 ----------
dataset = pd.read_parquet(fc.PROCESSED / "dataset_hourly_2021_2025q3.parquet")
naive = pd.read_parquet(fc.PROCESSED / "naive_forecasts_2021_2025q3.parquet")

NAIVE_COLUMNS = ["naive_d1", "naive_d7", "naive_mean7", "naive_epf"]
dataset = dataset.merge(naive[["ts_utc"] + NAIVE_COLUMNS], on="ts_utc",
                        how="left", validate="one_to_one")

# The models LEAR will be compared with, already computed in 08 with the SAME
# protocol (walk-forward, refit every 30 days): no need to re-run them here
xgb = pd.read_parquet(fc.PROCESSED / "xgb_validation_forecasts.parquet")
REFERENCE_COLUMNS = ["linear_v3_refit30", "linear_v3_avg_refit30", "xgb_single"]
dataset = dataset.merge(xgb[["ts_utc"] + REFERENCE_COLUMNS], on="ts_utc",
                        how="left", validate="one_to_one")

dataset = fc.add_split_columns(dataset)
dataset = fc.add_calendar_features(dataset)

# --- Checks ----------------------------------------------------------------------
assert len(dataset) == 41615
assert dataset.loc[dataset["split"] == "validation",
                   NAIVE_COLUMNS + REFERENCE_COLUMNS].notna().all().all()
print("dataset ready:", dataset.shape)

dataset ready: (41615, 29)


In [2]:
# --- The LEAR design matrix: full daily profiles, not single hours -----------------
# One row per local day, one column per local hour, same DST convention as 06:
# 25-hour days average the two 02:00 rows, 23-hour days interpolate the missing one.
def daily_profile(df: pd.DataFrame, column: str) -> pd.DataFrame:
    profile = df.pivot_table(index="local_date", columns="local_hour",
                             values=column, aggfunc="mean")
    profile = profile.T.interpolate().T
    assert profile.shape == (1734, 24), profile.shape
    assert profile.notna().all().all(), f"holes in the daily profile of {column}"
    # shift(k) means "k days earlier" only if no calendar day is missing
    day_steps = profile.index.to_series().diff().dropna()
    assert (day_steps == pd.Timedelta(days=1)).all(), "a calendar day is missing"
    return profile


def lagged_block(profile: pd.DataFrame, lags: list, prefix: str) -> pd.DataFrame:
    """Flatten `profile` shifted by each lag into columns <prefix>_d<lag>_h<hour>."""
    columns = {}
    for lag in lags:
        shifted = profile.shift(lag)
        for hour in range(24):
            columns[f"{prefix}_d{lag}_h{hour:02d}"] = shifted[hour]
    return pd.DataFrame(columns, index=profile.index)


PRICE_LAGS = [1, 2, 3, 7]   # Lago, Marcjasz, De Schutter & Weron (2021)
EXO_LAGS = [0, 1, 7]        # lag 0 = day D itself: forecasts known at 12:00 D-1

lag_matrix = pd.concat([
    lagged_block(daily_profile(dataset, TARGET), PRICE_LAGS, "p"),
    lagged_block(daily_profile(dataset, "load_forecast_mw"), EXO_LAGS, "load"),
    lagged_block(daily_profile(dataset, "residual_load_mw"), EXO_LAGS, "rl"),
], axis=1)

# Gas is a DAILY price, so it needs no hourly profile. gas_d1 is the price already
# attached to every hour of day D in 05 (the D-1 gas day); gas_d7 is the one of a
# week earlier, i.e. six days before that.
gas_daily = dataset.groupby("local_date")["gas_price_d1_eur_mwh"].first()
lag_matrix["gas_d1"] = gas_daily
lag_matrix["gas_d7"] = gas_daily.shift(6)

PRICE_COLUMNS = [c for c in lag_matrix.columns if c.startswith("p_")]
EXO_COLUMNS = [c for c in lag_matrix.columns if c.startswith(("load_", "rl_"))]
GAS_COLUMNS = ["gas_d1", "gas_d7"]

dataset = dataset.merge(lag_matrix, left_on="local_date", right_index=True,
                        how="left", validate="many_to_one")

# --- Checks -------------------------------------------------------------------------
first_complete = lag_matrix.dropna().index.min()
print(f"{len(PRICE_COLUMNS)} price + {len(EXO_COLUMNS)} exogenous + {len(GAS_COLUMNS)} gas "
      f"= {lag_matrix.shape[1]} columns; complete from {first_complete.date()}")

# p_d1_h19 must be yesterday's 19:00 price, i.e. exactly what naive_d1 uses
check_day = dataset.loc[dataset["local_date"] == pd.Timestamp("2024-05-15")]
previous_19 = dataset.loc[(dataset["local_date"] == pd.Timestamp("2024-05-14"))
                          & (dataset["local_hour"] == 19), TARGET]
assert np.allclose(check_day["p_d1_h19"].unique(), previous_19.to_numpy())

# No holes anywhere we will ever fit or score (only the first week of 2021 is NaN)
assert dataset.loc[dataset["split"] != "train",
                   PRICE_COLUMNS + EXO_COLUMNS + GAS_COLUMNS].notna().all().all()
print("lag matrix checks: OK")

96 price + 144 exogenous + 2 gas = 242 columns; complete from 2021-01-08
lag matrix checks: OK


In [3]:
# --- The LEAR: 24 hourly LASSO regressions ------------------------------------------
from sklearn.linear_model import LassoLarsIC

FEATURES_V1 = PRICE_COLUMNS + EXO_COLUMNS   # faithful LEAR: no gas yet


def make_lear(features: list, criterion: str = "aic"):
    """Return (fit, predict, design) for the LEAR of Lago et al. (2021).

    Three ingredients, all estimated on the training window only:
      1. asinh((price - median) / MAD) on the target AND on every lagged price:
         it compresses spikes without the zero-price problem of a log;
      2. standardisation of every column: the LASSO penalty is a SINGLE number
         for all coefficients, so the columns must share a scale, otherwise the
         cheapest ones to shrink are simply those measured in large units
         (MW vs EUR/MWh). Unlike OLS, a LASSO is not scale-invariant;
      3. LassoLarsIC: the whole LASSO path is computed with LARS and the penalty
         alpha is picked by an information criterion. No cross-validation, which
         on a time series would need blocking anyway.
    Day-of-week dummies are appended AFTER standardisation and stay 0/1,
    Monday = reference, exactly as in 07.
    """
    price_index = [i for i, col in enumerate(features) if col.startswith("p_d")]

    def design(df: pd.DataFrame, models: dict | None = None) -> np.ndarray:
        X = df[features].astype(float).to_numpy()
        if models is not None:
            X[:, price_index] = np.arcsinh(
                (X[:, price_index] - models["median"]) / models["mad"])
            X = (X - models["mu"]) / models["sd"]
        dummies = np.stack([(df["day_of_week"].to_numpy() == d).astype(float)
                            for d in range(1, 7)], axis=1)
        return np.hstack([X, dummies])

    def fit(train: pd.DataFrame) -> dict:
        train = train.dropna(subset=features + [TARGET])   # first week of 2021 has no lags
        median = train[TARGET].median()
        mad = (train[TARGET] - median).abs().median()      # median absolute deviation
        models = {"median": median, "mad": mad}

        # Scaling constants from the asinh-ed columns of THIS window only
        X_raw = train[features].astype(float).to_numpy()
        X_raw[:, price_index] = np.arcsinh((X_raw[:, price_index] - median) / mad)
        models["mu"], models["sd"] = X_raw.mean(axis=0), X_raw.std(axis=0)
        models["sd"][models["sd"] == 0] = 1.0   # a constant column would divide by zero

        X = design(train, models)
        y = np.arcsinh((train[TARGET].to_numpy() - median) / mad)
        hours = train["local_hour"].to_numpy()
        for hour in range(24):
            rows = hours == hour
            models[hour] = LassoLarsIC(criterion=criterion).fit(X[rows], y[rows])
        return models

    def predict(models: dict, day: pd.DataFrame) -> np.ndarray:
        X = design(day, models)
        hours = day["local_hour"].to_numpy()
        forecast = np.empty(len(day))
        for hour in np.unique(hours):
            rows = hours == hour
            z = models[hour].predict(X[rows])
            forecast[rows] = np.sinh(z) * models["mad"] + models["median"]   # back to EUR/MWh
        return forecast

    return fit, predict, design


# --- What does the LASSO keep? Fit on the train split (= the model of 1 Oct 2023) ---
import time

fit_v1, predict_v1, design_v1 = make_lear(FEATURES_V1)
train = dataset.loc[dataset["split"] == "train"]

start = time.perf_counter()
models_v1 = fit_v1(train)
print(f"24 hourly LASSO fitted in {time.perf_counter() - start:.1f} s")

FEATURE_NAMES = FEATURES_V1 + [f"dow_{d}" for d in range(1, 7)]
coefficients = pd.DataFrame({hour: models_v1[hour].coef_ for hour in range(24)},
                            index=FEATURE_NAMES)

n_kept = (coefficients != 0).sum()
print(f"\nnon-zero coefficients out of {len(FEATURE_NAMES)}: "
      f"min {n_kept.min()}, median {int(n_kept.median())}, max {n_kept.max()}")

# Coefficients live in standardised asinh space, so their sizes ARE comparable:
# "one standard deviation of this column moves the transformed price by X"
for hour in [4, 19]:
    kept = coefficients[hour][coefficients[hour] != 0].abs().sort_values(ascending=False)
    print(f"\nhour {hour:02d}: {len(kept)} features kept, 10 largest |coefficient|")
    print(coefficients.loc[kept.index[:10], hour].round(3).to_string())

24 hourly LASSO fitted in 1.6 s

non-zero coefficients out of 246: min 44, median 90, max 138

hour 04: 138 features kept, 10 largest |coefficient|
p_d1_h23     0.989
p_d1_h05     0.303
p_d3_h05    -0.279
p_d1_h00    -0.230
p_d3_h03     0.219
p_d1_h01     0.217
rl_d0_h05    0.213
p_d1_h02    -0.212
p_d7_h08     0.179
p_d1_h22    -0.174

hour 19: 44 features kept, 10 largest |coefficient|
p_d1_h19     0.446
p_d2_h19     0.266
p_d1_h22     0.162
rl_d0_h20    0.127
p_d7_h08     0.098
p_d3_h23     0.079
p_d7_h00    -0.059
p_d7_h23    -0.059
rl_d0_h09   -0.054
rl_d0_h23    0.053


In [4]:
# --- Walk-forward on the validation year: refit every 30 days, expanding window ------
# Same protocol as xgb_single in 08, so the comparison is like-for-like. The
# features are always rebuilt for every day: only the COEFFICIENTS are 30 days old.
REFIT_DAYS = 30

start = time.perf_counter()
dataset["lear_v1"] = fc.walk_forward(
    dataset, fit_v1, predict_v1, split="validation", refit_every_days=REFIT_DAYS,
)
print(f"walk-forward done in {time.perf_counter() - start:.0f} s\n")

# --- Metrics against the naive and the two models already chosen --------------------
results_v1 = fc.evaluate(dataset, ["naive_epf"] + REFERENCE_COLUMNS + ["lear_v1"],
                         split="validation")
print(results_v1[results_v1["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "WAPE_pct", "rMAE"]].round(2).to_string())

# --- Significance: is LEAR better than the linear control and than XGBoost? ----------
print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, reference, "lear_v1", split="validation", loss=loss)
    for reference in ["naive_epf", "linear_v3_refit30", "xgb_single"]
    for loss in ["abs", "squared"]
]).drop(columns=["forecast_b", "n_days"]).round(4).to_string(index=False))

walk-forward done in 20 s

                         MAE   RMSE  WAPE_pct  rMAE
forecast                                           
naive_epf              12.30  18.35     11.43  1.00
linear_v3_refit30       9.70  13.58      9.02  0.79
linear_v3_avg_refit30   9.42  13.20      8.75  0.77
xgb_single              9.11  12.94      8.46  0.74
lear_v1                 9.30  12.85      8.64  0.76

       forecast_a    loss  mean_daily_diff  dm_stat  p_value_b_better
        naive_epf     abs           3.0008   8.6703            0.0000
        naive_epf squared         171.8291   7.5204            0.0000
linear_v3_refit30     abs           0.4077   1.8045            0.0356
linear_v3_refit30 squared          19.4127   2.0576            0.0198
       xgb_single     abs          -0.1914  -0.9794            0.8363
       xgb_single squared           2.4268   0.2980            0.3828


In [5]:
# --- Modification A: add the gas price ------------------------------------------------
# LEAR has no fuel cost at all: the marginal unit is almost always a CCGT, so in
# principle this is the single most motivated feature we can add. gas_d1 is the
# gas day already attached to day D; gas_d7 lets the model see the recent trend.
FEATURES_V2 = FEATURES_V1 + GAS_COLUMNS
fit_v2, predict_v2, design_v2 = make_lear(FEATURES_V2)

start = time.perf_counter()
dataset["lear_v2"] = fc.walk_forward(
    dataset, fit_v2, predict_v2, split="validation", refit_every_days=REFIT_DAYS,
)
print(f"walk-forward done in {time.perf_counter() - start:.0f} s\n")

results_v2 = fc.evaluate(dataset, ["naive_epf"] + REFERENCE_COLUMNS + ["lear_v1", "lear_v2"],
                         split="validation")
print(results_v2[results_v2["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "rMAE"]].round(2).to_string())

print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, a, b, split="validation", loss=loss)
    for a, b in [("lear_v1", "lear_v2"),
                 ("linear_v3_refit30", "lear_v2"),
                 ("xgb_single", "lear_v2")]
    for loss in ["abs", "squared"]
]).drop(columns=["n_days"]).round(4).to_string(index=False))

# --- Does gas help WHERE it is supposed to? -------------------------------------------
# Same logic as `yesterday_holiday` in 07: a feature motivated by a mechanism must
# show its effect on the days that mechanism is about. Lagged prices already contain
# the gas level; what they CANNOT contain is a gas move that happened yesterday.
gas_daily = dataset.groupby("local_date")["gas_price_d1_eur_mwh"].first()
dataset = dataset.merge(gas_daily.diff().abs().rename("gas_move"),
                        left_on="local_date", right_index=True, how="left")

validation = dataset.loc[dataset["split"] == "validation"]
threshold = validation.groupby("local_date")["gas_move"].first().quantile(0.9)
big_move = validation["gas_move"] >= threshold
print(f"\n|daily gas change| 90th percentile: {threshold:.2f} EUR/MWh "
      f"({validation.loc[big_move, 'local_date'].nunique()} days of "
      f"{validation['local_date'].nunique()})")
print(pd.DataFrame({
    col: {
        "big gas move": (validation[TARGET] - validation[col])[big_move].abs().mean(),
        "other days": (validation[TARGET] - validation[col])[~big_move].abs().mean(),
    }
    for col in ["lear_v1", "lear_v2", "xgb_single"]
}).round(2).to_string())

walk-forward done in 20 s

                         MAE   RMSE  rMAE
forecast                                 
naive_epf              12.30  18.35  1.00
linear_v3_refit30       9.70  13.58  0.79
linear_v3_avg_refit30   9.42  13.20  0.77
xgb_single              9.11  12.94  0.74
lear_v1                 9.30  12.85  0.76
lear_v2                 9.24  12.76  0.75

       forecast_a forecast_b    loss  mean_daily_diff  dm_stat  p_value_b_better
          lear_v1    lear_v2     abs           0.0567   1.2941            0.0978
          lear_v1    lear_v2 squared           2.2638   1.2778            0.1007
linear_v3_refit30    lear_v2     abs           0.4644   2.1012            0.0178
linear_v3_refit30    lear_v2 squared          21.6765   2.3881            0.0085
       xgb_single    lear_v2     abs          -0.1347  -0.7184            0.7637
       xgb_single    lear_v2 squared           4.6906   0.6170            0.2686

|daily gas change| 90th percentile: 2.12 EUR/MWh (37 days of 366)
  

In [6]:
# --- Modification D: calibration window averaging ------------------------------------
# Windows SHORTER than the number of features are impossible here: LassoLarsIC needs
# n_samples > n_features to estimate the noise variance used by the AIC, and each
# hourly model sees only one row per day. With 248 columns, 365 days is the shortest
# usable window -- the 56/84-day windows of epftoolbox are out of reach.
for window in [365, 730]:
    start = time.perf_counter()
    dataset[f"lear_v1_roll{window}"] = fc.walk_forward(
        dataset, fit_v1, predict_v1, split="validation",
        window_days=window, refit_every_days=REFIT_DAYS,
    )
    print(f"rolling {window} days: {time.perf_counter() - start:.0f} s")

# The two-window average is the recipe already used in 07 and 08 (expanding + 365),
# decided before looking at these numbers. The three-window version is a check.
dataset["lear_v1_avg"] = dataset[["lear_v1", "lear_v1_roll365"]].mean(axis=1)
dataset["lear_v1_avg3"] = dataset[["lear_v1", "lear_v1_roll365", "lear_v1_roll730"]].mean(axis=1)

WINDOW_COLUMNS = ["lear_v1", "lear_v1_roll365", "lear_v1_roll730",
                  "lear_v1_avg", "lear_v1_avg3"]
results_d = fc.evaluate(dataset, ["naive_epf", "linear_v3_avg_refit30", "xgb_single"]
                        + WINDOW_COLUMNS, split="validation")
print()
print(results_d[results_d["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "rMAE"]].round(3).to_string())

# --- Significance ----------------------------------------------------------------------
print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, a, b, split="validation", loss=loss)
    for a, b in [("lear_v1", "lear_v1_roll365"),
                 ("lear_v1", "lear_v1_roll730"),
                 ("lear_v1", "lear_v1_avg"),
                 ("lear_v1_avg3", "lear_v1_avg"),
                 ("linear_v3_avg_refit30", "lear_v1_avg"),
                 ("xgb_single", "lear_v1_avg")]
    for loss in ["abs", "squared"]
]).drop(columns=["n_days"]).round(4).to_string(index=False))

# --- Why does averaging work? Look at the ERRORS, not at the forecasts ----------------
# Forecasts of good models are all correlated ~0.99 with each other; what decides
# whether combining helps is how much their ERRORS have in common.
validation = dataset.loc[dataset["split"] == "validation"]
errors = validation[["lear_v1", "lear_v1_roll365", "xgb_single", "linear_v3_avg_refit30"]] \
    .apply(lambda column: validation[TARGET] - column)
print("\nerror correlation:")
print(errors.corr().round(3).to_string())

rolling 365 days: 21 s
rolling 730 days: 20 s

                          MAE    RMSE   rMAE
forecast                                    
naive_epf              12.296  18.352  1.000
linear_v3_avg_refit30   9.416  13.198  0.766
xgb_single              9.105  12.942  0.740
lear_v1                 9.296  12.846  0.756
lear_v1_roll365         9.586  14.262  0.780
lear_v1_roll730        10.361  14.165  0.843
lear_v1_avg             9.030  12.891  0.734
lear_v1_avg3            9.279  12.991  0.755

           forecast_a      forecast_b    loss  mean_daily_diff  dm_stat  p_value_b_better
              lear_v1 lear_v1_roll365     abs          -0.2877  -1.6493            0.9505
              lear_v1 lear_v1_roll365 squared         -38.2154  -2.1668            0.9849
              lear_v1 lear_v1_roll730     abs          -1.0642  -8.3356            1.0000
              lear_v1 lear_v1_roll730 squared         -35.5654  -7.2635            1.0000
              lear_v1     lear_v1_avg     abs       

In [7]:
# --- Persist validation forecasts, metrics, DM tests and the frozen config -----------
import json

LEAR_VERSIONS = ["lear_v1", "lear_v2",
                 "lear_v1_roll365", "lear_v1_roll730",
                 "lear_v1_avg", "lear_v1_avg3"]
FINAL_LEAR = "lear_v1_avg"          # chosen on validation; the test set comes later, once
AVERAGED_WINDOWS = [None, 365]      # None = expanding

fc.RESULTS.mkdir(exist_ok=True)

# Validation forecasts of every version, for the final comparison notebook
validation = dataset.loc[dataset["split"] == "validation"]
KEEP = ["ts_utc", "ts_local", "local_date", "local_hour", "period", "split", TARGET]
forecasts_out = validation[KEEP + NAIVE_COLUMNS + LEAR_VERSIONS].reset_index(drop=True)

forecasts_path = fc.PROCESSED / "lear_validation_forecasts.parquet"
forecasts_out.to_parquet(forecasts_path, index=False)
pd.testing.assert_frame_equal(pd.read_parquet(forecasts_path), forecasts_out)

metrics = fc.evaluate(dataset, NAIVE_COLUMNS + LEAR_VERSIONS, split="validation")
metrics.to_csv(fc.RESULTS / "09_lear_validation_metrics.csv", index=False)

# Every step tested against the model it modified, plus the cross-model comparisons
STEPS = [
    ("naive_epf", "lear_v1"),
    ("linear_v3_refit30", "lear_v1"),        # same protocol, much less information
    ("xgb_single", "lear_v1"),
    ("lear_v1", "lear_v2"),                  # A: gas -> not significant, rejected
    ("lear_v1", "lear_v1_roll365"),          # D: rolling windows
    ("lear_v1", "lear_v1_roll730"),
    ("lear_v1", "lear_v1_avg"),              # D: window averaging
    ("lear_v1_avg3", "lear_v1_avg"),         # adding the 730-day window hurts
    ("linear_v3_avg_refit30", "lear_v1_avg"),
    ("xgb_single", "lear_v1_avg"),
]
dm_steps = pd.DataFrame([
    fc.diebold_mariano(dataset, a, b, split="validation", loss=loss)
    for a, b in STEPS for loss in ["abs", "squared"]
])
dm_steps.to_csv(fc.RESULTS / "09_lear_dm_tests.csv", index=False)

# Frozen recipe: enough to rebuild the exact model in the final notebook
config = {
    "name": FINAL_LEAR,
    "price_lags": PRICE_LAGS,
    "exo_lags": EXO_LAGS,
    "exogenous": ["load_forecast_mw", "residual_load_mw"],
    "gas": False,
    "n_features": len(FEATURES_V1) + 6,          # + day-of-week dummies
    "transform": "asinh((price - median) / MAD), window median and MAD",
    "estimator": "LassoLarsIC(criterion='aic')",
    "per_hour": True,
    "refit_every_days": REFIT_DAYS,
    "averaged_windows": AVERAGED_WINDOWS,
}
(fc.RESULTS / "09_lear_config.json").write_text(json.dumps(config, indent=2))

print(f"saved {forecasts_out.shape} -> {forecasts_path.name}")
print(f"saved {metrics.shape} -> results/09_lear_validation_metrics.csv")
print(f"saved {dm_steps.shape} -> results/09_lear_dm_tests.csv")
print(f"\nfinal LEAR: {FINAL_LEAR}")
print(metrics[(metrics["period"] == "all") & (metrics["forecast"] == FINAL_LEAR)]
      [["MAE", "RMSE", "WAPE_pct", "rMAE"]].round(3).to_string(index=False))

saved (8784, 17) -> lear_validation_forecasts.parquet
saved (20, 8) -> results/09_lear_validation_metrics.csv
saved (20, 7) -> results/09_lear_dm_tests.csv

final LEAR: lear_v1_avg
 MAE   RMSE  WAPE_pct  rMAE
9.03 12.891     8.392 0.734


## Summary

**Model.** LEAR (Lago, Marcjasz, De Schutter & Weron, 2021): 24 hourly LASSO
regressions on the full daily price profiles of D-1, D-2, D-3 and D-7 (96 lagged
prices) plus the load and residual-load profiles of D, D-1 and D-7, with
day-of-week dummies — 248 columns per hourly model. Prices are transformed with
asinh((p − median) / MAD) on the training window, every column is standardised,
and the penalty is chosen by AIC (`LassoLarsIC`). Walk-forward over the validation
year with a refit every 30 days, the same protocol as `xgb_single` in 08.
**The test set has not been looked at.**

| Version | Change | MAE | RMSE | rMAE | DM (abs) vs previous |
|---|---|---|---|---|---|
| `lear_v1` | faithful LEAR, expanding window | 9.30 | 12.85 | 0.756 | vs `linear_v3_refit30` 1.80 (p 0.036) |
| `lear_v2` | **A** + gas D-1 and D-7 | 9.24 | 12.76 | 0.751 | 1.29 (p 0.098, n.s.) — rejected |
| `lear_v1_roll365` | **D** rolling 365-day window | 9.59 | 14.26 | 0.780 | −1.65 → worse |
| `lear_v1_roll730` | **D** rolling 730-day window | 10.36 | 14.17 | 0.843 | −8.34 → much worse |
| **`lear_v1_avg`** | **D** mean of expanding and rolling 365 | **9.03** | 12.89 | **0.734** | **2.89 (p 0.002)** ← chosen |
| `lear_v1_avg3` | **D** also averaging the 730-day window | 9.28 | 12.99 | 0.755 | −5.13 → worse than `lear_v1_avg` |

**Final choice: `lear_v1_avg`** (rMAE 0.734, the best MAE on validation so far;
statistically tied with `xgb_single`: DM 0.40, p 0.34).

**What each step taught:**
- **Short calibration windows are impossible here.** `LassoLarsIC` needs
  n_samples > n_features to